# Detecting industrial defects from normal images

Defect examples are often harder to collect than images of products in good condition. This project asks whether a model can learn what a normal product looks like and flag images that differ from it.

I compare three approaches on the bottle, metal nut and tile categories of MVTec AD: nearest-neighbour matching with pretrained image features, a convolutional autoencoder, and a patch-based detector inspired by PatchCore. The comparison covers both image-level detection and, where available, the location of defects.

The main experiment uses one fixed split and one seed. An optional study repeats the comparison with different amounts of training data and three seeds. Results are generated by the notebook; no scores have been filled in beforehand.


## Setup

This version runs without notebook Internet access. It reads two attached inputs:

- **MVTec AD**, with extracted bottle, metal_nut and tile folders. Each category needs train, test and ground_truth.
- **resnet18-f37072fd.pth**, the ImageNet-pretrained ResNet18 weights.

The first cell locates both inputs. I leave the paths on automatic detection unless more than one copy is attached. A GPU is useful, but the code also runs on a CPU.

The default is `RUN_MODE = "standard"`. For a short setup check, I use `"quick"`; those results are not the final experiment. The `"study"` setting runs the training-data comparison and takes longer.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
DATA_ROOT = None  # Optional parent directory containing the category folders.
WEIGHTS_PATH = None  # Optional full path to the uploaded .pth file.
REQUIRED_CATEGORIES = ('bottle', 'metal_nut', 'tile')

def discover_mvtec(input_root, categories, data_root=None):
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise RuntimeError('No /kaggle/input directory. Run this notebook in Kaggle and attach MVTec AD.')
    mounts = sorted(p for p in input_root.iterdir() if p.is_dir())
    print('Attached input locations:')
    for mount in mounts:
        print(' -', mount)
    roots = {}
    issues = []
    for category in categories:
        candidates = ([Path(data_root) / category] if data_root is not None else
                      [p for p in input_root.rglob(category) if p.is_dir()])
        valid = sorted({p for p in candidates if all((p / sub).is_dir() for sub in
                        ('train/good', 'test/good', 'ground_truth'))})
        if len(valid) == 1:
            roots[category] = valid[0]
            print(f'FOUND {category}: {valid[0]}')
        elif not valid:
            issues.append(f'{category}: missing train/good, test/good or ground_truth')
        else:
            issues.append(f'{category}: multiple copies found: {valid}; set DATA_ROOT to one parent')
    if issues:
        raise RuntimeError(
            'MVTec AD input check failed:\n' + '\n'.join(issues) +
            '\n\nIf the sidebar only shows NOTEBOOKS, attach the actual MVTec AD DATASET. '
            'Open the source notebook Input tab, click MVTec AD under Data Sources, '
            'then add that dataset to this notebook. The images must be extracted, '
            'not only stored inside a ZIP/TAR archive. Re-run this cell after attaching it.')
    return roots

CATEGORY_ROOTS = discover_mvtec(INPUT_ROOT, REQUIRED_CATEGORIES, DATA_ROOT)
print('Dataset structure check passed. Continue with the remaining cells.')

def discover_weights(input_root, explicit_path=None):
    if explicit_path is not None:
        path = Path(explicit_path)
        if not path.is_file():
            raise FileNotFoundError(f'Weights file not found: {path}')
        return str(path)
    candidates = sorted(p for p in Path(input_root).rglob('resnet18-f37072fd*.pth') if p.is_file())
    if not candidates:
        raise FileNotFoundError(
            'Attach the uploaded resnet18-f37072fd.pth file as a Kaggle input, then rerun this cell. '
            'This notebook does not download weights from the Internet.')
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple weights files found: {candidates}. Set WEIGHTS_PATH to the intended file.')
    return str(candidates[0])

LOCAL_WEIGHTS = discover_weights(INPUT_ROOT, WEIGHTS_PATH)
print('Pretrained weights:', LOCAL_WEIGHTS)
print('Both inputs are ready. No Internet download is needed.')


In [ ]:
from pathlib import Path
import copy, gc, json, platform, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score, confusion_matrix)
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from tqdm.auto import tqdm

RUN_MODE = "standard"          # quick | standard | study

assert RUN_MODE in {"quick", "standard", "study"}
CATEGORIES = ["bottle"] if RUN_MODE == "quick" else ["bottle", "metal_nut", "tile"]
FRACTIONS = [0.25] if RUN_MODE == "quick" else ([0.1, 0.25, 0.5, 1.0] if RUN_MODE == "study" else [1.0])
SEEDS = [42, 123, 2026] if RUN_MODE == "study" else [42]
EPOCHS = 3 if RUN_MODE == "quick" else 20
IMAGE_SIZE, MAP_SIZE, PATCH_GRID = 224, 128, 14
BATCH_SIZE, PATIENCE = 16, 4
CANDIDATE_CAP, BANK_SIZE, PATCHES_PER_IMAGE = 3000, 256, 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/kaggle/working/anomaly_results")
OUT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print("Device:", DEVICE, "| PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Mode:", RUN_MODE, "| categories:", CATEGORIES, "| fractions:", FRACTIONS, "| seeds:", SEEDS)
if RUN_MODE == "quick":
    print("SMOKE TEST ONLY: these settings are not my final experiment.")
if DEVICE.type == "cpu":
    warnings.warn("CPU selected. Start with quick mode to check setup.")


## Keeping training and evaluation separate

I keep the official test set unchanged. Within each category, I split the normal training images into 70% for fitting, 15% for validation and 15% for threshold calibration. The split uses seed 42 and stays fixed across experiments.

The autoencoder stops according to its validation reconstruction loss. I set each detector's threshold to the 95th percentile of its scores on the normal calibration images. Test defects are not used to choose a threshold or stop training. This gives an empirical operating point; it does not guarantee a 5% false-alarm rate on new data.

Images are resized to 224 × 224 without cropping. Localisation is evaluated at 128 × 128 to keep memory use manageable. These pixel scores should not be compared directly with benchmark scores measured at the original resolution.

In the optional data study, fractions refer to the fitting pool only. Smaller subsets are nested within larger ones for each seed. Validation and calibration images stay the same. The full study has 36 category, fraction and seed combinations.


In [ ]:
EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def images_in(folder):
    return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)

def locate_category(category):
    if category not in CATEGORY_ROOTS:
        raise RuntimeError(f'{category} was not checked. Add it to REQUIRED_CATEGORIES and rerun the input check.')
    return CATEGORY_ROOTS[category]

def make_records(category):
    root = locate_category(category)
    normal = images_in(root/"train/good")
    if len(normal) < 30:
        raise ValueError(f"Too few normal training images in {root}")
    order = np.random.default_rng(42).permutation(len(normal))
    nfit, nval = int(len(normal)*0.70), int(len(normal)*0.15)
    splits = {"fit": order[:nfit], "validation": order[nfit:nfit+nval],
              "calibration": order[nfit+nval:]}
    train = {split: [{"path": str(normal[i]), "label": 0, "defect": "good", "mask": None}
                     for i in indices] for split, indices in splits.items()}
    test = []
    for folder in sorted((root/"test").iterdir()):
        if not folder.is_dir():
            continue
        for p in images_in(folder):
            label = int(folder.name != "good")
            mask = root/"ground_truth"/folder.name/(p.stem+"_mask.png") if label else None
            if mask is not None and not mask.is_file():
                raise FileNotFoundError(f"Missing defect mask: {mask}")
            test.append({"path": str(p), "label": label, "defect": folder.name,
                         "mask": str(mask) if mask else None})
    assert set(r["label"] for r in test) == {0, 1}, "Test set must contain normal and defective images."
    all_paths = [r["path"] for group in [*train.values(), test] for r in group]
    assert len(all_paths) == len(set(all_paths)), "Overlapping file paths between splits."
    return train, test

class ImageDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        with Image.open(self.records[i]["path"]) as im:
            arr = np.asarray(im.convert("RGB").resize(
                (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR), dtype=np.float32)/255.
        return torch.from_numpy(arr.copy()).permute(2, 0, 1)

def loader(records, shuffle=False, seed=42):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(ImageDataset(records), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, generator=generator)

def load_masks(records):
    masks = []
    for r in records:
        if r["mask"] is None:
            masks.append(np.zeros((MAP_SIZE, MAP_SIZE), dtype=np.uint8))
        else:
            with Image.open(r["mask"]) as im:
                mask = np.asarray(im.convert("L").resize((MAP_SIZE, MAP_SIZE),
                                                        Image.Resampling.NEAREST))
            masks.append((mask > 0).astype(np.uint8))
    return np.stack(masks)

DATA = {category: make_records(category) for category in CATEGORIES}
display(pd.DataFrame([{"category": c, **{k: len(v) for k, v in tr.items()},
                       "test_normal": sum(r["label"] == 0 for r in te),
                       "test_defective": sum(r["label"] == 1 for r in te)}
                      for c, (tr, te) in DATA.items()]))


## Models

### Image-feature baseline

I extract a vector from the final convolutional layer of an ImageNet-pretrained ResNet18 and normalise it to unit length. An image receives a high anomaly score when its nearest normal training image is far away in this feature space. This is a useful baseline, although it cannot show where a defect is located.

### Autoencoder

The autoencoder learns to reconstruct normal images. I train it from scratch using mean squared reconstruction error and retain the weights with the lowest validation loss. At test time, I average absolute reconstruction error across colour channels to produce a spatial error map.

### Patch-based detector

For the third method, I combine intermediate ResNet18 features and compare local patches with a memory bank of normal patches. I sample candidate patches and use greedy farthest-point selection in a random projection to keep the bank small.

This is a **PatchCore-inspired implementation**, not a reproduction of the original benchmark. It uses ResNet18, a 14 × 14 patch grid, capped sampling, a 64-dimensional projection and no official image-score reweighting. For both spatial methods, I smooth the map and average its highest-scoring 1% of pixels to obtain an image score.

The feature methods start with ImageNet pretraining, whereas the autoencoder starts from scratch. That difference matters when interpreting the results: this compares complete approaches, not architectures with equal training histories.


In [ ]:
def load_backbone():
    # Initialise the architecture locally, then load the uploaded pretrained state.
    # weights=None here does not leave the model randomly initialised.
    net = resnet18(weights=None)
    state = torch.load(LOCAL_WEIGHTS, map_location="cpu", weights_only=True)
    net.load_state_dict(state, strict=True)
    return net.to(DEVICE).eval().requires_grad_(False)

BACKBONE = load_backbone()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE)[None, :, None, None]
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE)[None, :, None, None]

@torch.inference_mode()
def extract_features(records):
    global_features, patches = [], []
    for x in loader(records):
        x = (x.to(DEVICE)-MEAN)/STD
        n = BACKBONE
        x = n.maxpool(n.relu(n.bn1(n.conv1(x))))
        x = n.layer1(x)
        a = n.layer2(x)
        b = n.layer3(a)
        g = F.normalize(F.adaptive_avg_pool2d(n.layer4(b), 1).flatten(1), dim=1)
        a = F.adaptive_avg_pool2d(F.avg_pool2d(a, 3, 1, 1), PATCH_GRID)
        b = F.adaptive_avg_pool2d(F.avg_pool2d(b, 3, 1, 1), PATCH_GRID)
        p = torch.cat([a, b], dim=1).flatten(2).transpose(1, 2)
        global_features.append(g.cpu())
        patches.append(p.cpu())
    return {"global": torch.cat(global_features), "patch": torch.cat(patches)}

def synchronise():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def nearest_distance(query, bank, chunk_size=1024):
    bank = bank.to(DEVICE)
    out = []
    for chunk in query.split(chunk_size):
        distances = torch.cdist(chunk.to(DEVICE), bank)
        out.append(distances.min(dim=1).values.cpu())
    return torch.cat(out)

def make_patch_bank(fit_patches, seed):
    rng = np.random.default_rng(seed)
    candidates = []
    for patches in fit_patches:
        idx = rng.choice(len(patches), min(PATCHES_PER_IMAGE, len(patches)), replace=False)
        candidates.append(patches[idx])
    candidates = torch.cat(candidates)
    if len(candidates) > CANDIDATE_CAP:
        candidates = candidates[rng.choice(len(candidates), CANDIDATE_CAP, replace=False)]
    generator = torch.Generator().manual_seed(seed)
    projection = torch.randn(candidates.shape[1], 64, generator=generator)/np.sqrt(64)
    z = (candidates @ projection).to(DEVICE)
    count = min(BANK_SIZE, len(z))
    current = int(((z-z.mean(0))**2).sum(1).argmax().item())
    minimum = torch.full((len(z),), float("inf"), device=DEVICE)
    selected = []
    for _ in range(count):
        selected.append(current)
        minimum = torch.minimum(minimum, ((z-z[current])**2).sum(1))
        minimum[selected] = -1
        current = int(minimum.argmax().item())
    return candidates[selected].contiguous()

def smooth_resize_maps(maps):
    maps = F.interpolate(maps[:, None], size=(MAP_SIZE, MAP_SIZE),
                         mode="bilinear", align_corners=False)[:, 0].cpu().numpy()
    return np.stack([gaussian_filter(m, sigma=2) for m in maps]).astype(np.float32)

def image_scores(maps):
    flat = maps.reshape(len(maps), -1)
    k = max(1, int(np.ceil(flat.shape[1]*0.01)))
    return np.partition(flat, -k, axis=1)[:, -k:].mean(axis=1)

def patch_predict(features, bank):
    n = len(features)
    distances = nearest_distance(features.reshape(-1, features.shape[-1]), bank)
    maps = smooth_resize_maps(distances.reshape(n, PATCH_GRID, PATCH_GRID))
    return image_scores(maps), maps

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,16,4,2,1), nn.ReLU(),
            nn.Conv2d(16,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,64,4,2,1), nn.ReLU())
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,16,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(16,3,4,2,1), nn.Sigmoid())
    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_autoencoder(fit, validation, seed):
    seed_everything(seed)
    model = Autoencoder().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss, stale, best_state, history = float("inf"), 0, None, []
    train_loader = loader(fit, shuffle=True, seed=seed)
    for epoch in range(1, EPOCHS+1):
        model.train()
        total = 0.
        for x in train_loader:
            x = x.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(x), x)
            loss.backward()
            optimizer.step()
            total += loss.item()*len(x)
        model.eval()
        val_total = 0.
        with torch.inference_mode():
            for x in loader(validation):
                x = x.to(DEVICE)
                val_total += F.mse_loss(model(x), x).item()*len(x)
        val_loss = val_total/len(validation)
        print(f"    Epoch {epoch:02d}/{EPOCHS}: train={total/len(fit):.6f}, validation={val_loss:.6f}", flush=True)
        history.append({"epoch": epoch, "train_mse": total/len(fit), "validation_mse": val_loss})
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model.eval(), history

@torch.inference_mode()
def autoencoder_predict(model, records):
    maps = []
    for x in loader(records):
        x = x.to(DEVICE)
        error = (model(x)-x).abs().mean(dim=1)
        maps.append(smooth_resize_maps(error))
    maps = np.concatenate(maps)
    return image_scores(maps), maps


## Evaluation

I report image-level area under the receiver operating characteristic curve (AUROC) and average precision (AP). At the threshold chosen from normal calibration images, I also report precision, recall, F1 and the false-positive rate. AP depends on how common defects are in the test set.

For the autoencoder and patch detector, pixel AUROC and AP are calculated across all test pixels, including pixels in normal images. Small defects can be poorly localised even when pooled pixel metrics look strong, so I inspect the maps as well.

Training times exclude shared feature extraction. Feature-method prediction times use cached embeddings, while autoencoder prediction time includes loading images and running the network. These timings are recorded for reference, not presented as a fair end-to-end speed benchmark.

Each run saves its scores, thresholds, split manifest and fitted model state. The image gallery includes a high-scoring normal image and a low-scoring defect under the patch detector, rather than showing only successful examples. Colour scales are fixed within each method and category, but differ between methods.


In [ ]:
def evaluate(scores, calibration_scores, records, maps=None, masks=None):
    y = np.array([r["label"] for r in records])
    threshold = float(np.quantile(calibration_scores, 0.95, method="higher"))
    predicted = (scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0,1]).ravel()
    result = {
        "image_auroc": roc_auc_score(y, scores),
        "image_ap": average_precision_score(y, scores),
        "precision": precision_score(y, predicted, zero_division=0),
        "recall": recall_score(y, predicted, zero_division=0),
        "f1": f1_score(y, predicted, zero_division=0),
        "test_fpr": fp/(fp+tn), "threshold": threshold,
        "calibration_fpr": float(np.mean(calibration_scores > threshold)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "pixel_auroc": np.nan, "pixel_ap": np.nan}
    if maps is not None:
        result["pixel_auroc"] = roc_auc_score(masks.reshape(-1), maps.reshape(-1))
        result["pixel_ap"] = average_precision_score(masks.reshape(-1), maps.reshape(-1))
    return result, predicted

def save_gallery(category, records, masks, ae_maps, patch_maps, patch_scores):
    normal = [i for i,r in enumerate(records) if r["label"] == 0]
    defective = [i for i,r in enumerate(records) if r["label"] == 1]
    # Deliberately inspect a high-scoring normal and a low-scoring defect too.
    indices = list(dict.fromkeys([normal[0], defective[0],
                  max(normal, key=lambda i: patch_scores[i]),
                  min(defective, key=lambda i: patch_scores[i])]))
    fig, axes = plt.subplots(len(indices), 4, figsize=(12, 3*len(indices)), squeeze=False)
    for row, i in enumerate(indices):
        with Image.open(records[i]["path"]) as im:
            axes[row,0].imshow(im.convert("RGB"))
        axes[row,0].set_title(f"{records[i]['defect']} / {Path(records[i]['path']).name}")
        axes[row,1].imshow(masks[i], cmap="gray", vmin=0, vmax=1)
        axes[row,1].set_title("Ground truth")
        for col, maps, title in [(2,ae_maps,"Autoencoder"), (3,patch_maps,"Patch detector")]:
            axes[row,col].imshow(maps[i], cmap="inferno", vmin=0, vmax=max(float(maps.max()),1e-8))
            axes[row,col].set_title(title)
        for ax in axes[row]:
            ax.axis("off")
    fig.suptitle(category + ": fixed scale within each method; scales differ between methods")
    fig.tight_layout()
    fig.savefig(OUT/f"{category}_error_gallery.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)

metrics_rows, prediction_rows, history_rows, manifest_rows = [], [], [], []
feature_timing = []
settings = dict(run_mode=RUN_MODE, categories=CATEGORIES, fractions=FRACTIONS, seeds=SEEDS,
                epochs=EPOCHS, split_seed=42, fit_share=0.7, validation_share=0.15,
                threshold_quantile=0.95, image_size=IMAGE_SIZE, map_size=MAP_SIZE,
                patch_grid=PATCH_GRID, candidate_cap=CANDIDATE_CAP, bank_size=BANK_SIZE,
                patches_per_image=PATCHES_PER_IMAGE, batch_size=BATCH_SIZE,
                pretrained_weights="ResNet18 ImageNet1K V1",
                source="https://www.mvtec.com/research-teaching/datasets/mvtec-ad")
(OUT/"settings.json").write_text(json.dumps(settings, indent=2))
(OUT/"environment.json").write_text(json.dumps({
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None
}, indent=2))

def flush_tables():
    for filename, rows in [("metrics", metrics_rows), ("test_predictions", prediction_rows),
                           ("training_history", history_rows), ("split_manifest", manifest_rows),
                           ("feature_extraction_times", feature_timing)]:
        pd.DataFrame(rows).to_csv(OUT/f"{filename}.csv", index=False)

for category, (train, test) in DATA.items():
    print("\nCategory:", category)
    masks = load_masks(test)
    synchronise()
    start = time.perf_counter()
    features = {split: extract_features(records) for split, records in
                {"fit": train["fit"], "calibration": train["calibration"], "test": test}.items()}
    synchronise()
    feature_timing.append({"category": category, "seconds": time.perf_counter()-start,
                           "scope": "fit + calibration + test; shared across runs"})
    for seed in SEEDS:
        ordering = np.random.default_rng(seed).permutation(len(train["fit"]))
        for fraction in FRACTIONS:
            selected = ordering[:max(2, int(np.ceil(len(ordering)*fraction)))]
            fit = [train["fit"][i] for i in selected]
            tag = f"{category}_seed{seed}_fraction{fraction:g}"
            run_dir = OUT/tag
            run_dir.mkdir(exist_ok=True)
            info = {"category": category, "seed": seed, "fraction": fraction, "n_fit": len(fit)}
            for split, records in {"fit":fit, "validation":train["validation"],
                                   "calibration":train["calibration"], "test":test}.items():
                manifest_rows.extend([{**info, "split":split, **r} for r in records])
            maps_by_method = {}
            scores_by_method = {}
            for method in ["global_nn", "autoencoder", "patchcore_inspired"]:
                print(" ", tag, method)
                synchronise()
                start = time.perf_counter()
                if method == "global_nn":
                    bank = features["fit"]["global"][selected].clone()
                    artifact = {"bank":bank}
                elif method == "patchcore_inspired":
                    bank = make_patch_bank(features["fit"]["patch"][selected], seed)
                    artifact = {"bank":bank}
                else:
                    model, history = train_autoencoder(fit, train["validation"], seed)
                    artifact = {"state_dict": {k:v.detach().cpu() for k,v in model.state_dict().items()}}
                    history_rows.extend([{**info, **h} for h in history])
                synchronise()
                fit_seconds = time.perf_counter()-start
                start = time.perf_counter()
                if method == "global_nn":
                    calibration = nearest_distance(features["calibration"]["global"], bank).numpy()
                    scores = nearest_distance(features["test"]["global"], bank).numpy()
                    maps = None
                elif method == "patchcore_inspired":
                    calibration, _ = patch_predict(features["calibration"]["patch"], bank)
                    scores, maps = patch_predict(features["test"]["patch"], bank)
                else:
                    calibration, _ = autoencoder_predict(model, train["calibration"])
                    scores, maps = autoencoder_predict(model, test)
                synchronise()
                prediction_seconds = time.perf_counter()-start
                result, predicted = evaluate(scores, calibration, test, maps, masks)
                metrics_rows.append({**info, "method":method, **result,
                                     "fit_seconds":fit_seconds,
                                     "calibration_plus_test_prediction_seconds":prediction_seconds})
                prediction_rows.extend([{**info, "method":method, **r, "score":float(s),
                                         "predicted":int(p), "threshold":result["threshold"]}
                                        for r,s,p in zip(test,scores,predicted)])
                artifact.update({"method":method, "threshold":result["threshold"],
                                 "settings":settings, "run":info})
                torch.save(artifact, run_dir/f"{method}.pt")
                flush_tables()
                maps_by_method[method] = maps
                scores_by_method[method] = scores
                if method == "autoencoder":
                    del model
            if seed == SEEDS[0] and fraction == FRACTIONS[-1]:
                save_gallery(category, test, masks, maps_by_method["autoencoder"],
                             maps_by_method["patchcore_inspired"], scores_by_method["patchcore_inspired"])
            del maps_by_method, scores_by_method
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
    del features
    gc.collect()

print("Saved results to:", OUT)


## Results

The tables below are calculated from the current run. I look at each category before taking a macro average, since a method can work well on one surface and struggle on another.

When reviewing the outputs, I check whether the image-level ranking agrees with localisation quality, which defects receive low scores, and which normal images trigger false alarms. The gallery gives a starting point for that analysis.

For the optional study, error bars show standard deviation across seeds. They describe variation in these runs, not confidence intervals for performance in a new factory. The fitting-data study also keeps validation and calibration sizes fixed, so it does not measure the total amount of normal data a deployment would need.


In [ ]:
results = pd.DataFrame(metrics_rows)
columns = ["category","seed","fraction","method","n_fit","image_auroc","image_ap",
           "recall","test_fpr","f1","pixel_auroc","pixel_ap"]
display(results[columns].round(4))
main = results[results["fraction"] == max(FRACTIONS)]
macro = main.groupby("method")[["image_auroc","image_ap","pixel_auroc","pixel_ap"]].mean()
macro.to_csv(OUT/"macro_average.csv")
display(macro.round(4))
print("Macro means weight each evaluated category/seed equally.")
print("Highest observed mean image AUROC:", macro["image_auroc"].idxmax())
print("This ranking describes this run; it is not evidence of universal superiority.")

if RUN_MODE == "study":
    summary = results.groupby(["category","fraction","method"])["image_auroc"].agg(["mean","std"]).reset_index()
    summary.to_csv(OUT/"fraction_study.csv", index=False)
    fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(5*len(CATEGORIES),4), squeeze=False)
    for ax, category in zip(axes[0], CATEGORIES):
        for method, frame in summary[summary["category"] == category].groupby("method"):
            ax.errorbar(frame["fraction"], frame["mean"], yerr=frame["std"].fillna(0),
                        marker="o", capsize=3, label=method)
        ax.set(title=category, xlabel="Fraction of fitting pool", ylabel="Image AUROC")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT/"training_fraction_study.png", dpi=150)
    plt.show()
    plt.close(fig)

history = pd.DataFrame(history_rows)
fig, ax = plt.subplots(figsize=(8,4))
for (category, seed, fraction), frame in history.groupby(["category","seed","fraction"]):
    if seed == SEEDS[0] and fraction == max(FRACTIONS):
        ax.plot(frame["epoch"], frame["validation_mse"], label=category)
ax.set(xlabel="Epoch", ylabel="Validation reconstruction MSE", title="Autoencoder early stopping")
ax.legend()
fig.tight_layout()
fig.savefig(OUT/"autoencoder_validation_loss.png", dpi=150)
plt.show()
plt.close(fig)


## Loading a saved detector

Each checkpoint contains the fitted autoencoder or memory bank, its decision threshold and the run settings. The feature methods also need the same ResNet18 weights and preprocessing used above. Those shared weights are not duplicated in every checkpoint.

The optional cell below scores one new image after the earlier definitions have been run. It uses the saved threshold without recalibration. I only load checkpoints from a trusted source.


In [ ]:
# Optional example: change to a real image path and checkpoint from this run.
NEW_IMAGE = None
CHECKPOINT = None

if NEW_IMAGE is not None and CHECKPOINT is not None:
    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    assert saved["settings"]["image_size"] == IMAGE_SIZE
    assert saved["settings"]["patch_grid"] == PATCH_GRID
    record = [{"path": str(NEW_IMAGE), "label": 0, "defect":"unknown", "mask":None}]
    method = saved["method"]
    if method == "autoencoder":
        restored = Autoencoder().to(DEVICE).eval()
        restored.load_state_dict(saved["state_dict"])
        scores, maps = autoencoder_predict(restored, record)
    else:
        encoded = extract_features(record)
        if method == "global_nn":
            scores = nearest_distance(encoded["global"], saved["bank"]).numpy()
            maps = None
        else:
            scores, maps = patch_predict(encoded["patch"], saved["bank"])
    print({"score": float(scores[0]), "threshold": saved["threshold"],
           "predicted_defect": bool(scores[0] > saved["threshold"])})
    if maps is not None:
        plt.imshow(maps[0], cmap="inferno")
        plt.title("Anomaly map; display scale is for this image only")
        plt.axis("off")
        plt.show()


## Limitations

This experiment covers three categories from one controlled dataset. It does not establish how the models would behave with a different camera, lighting setup or production line. Resizing may remove small defects, and pooled pixel metrics can hide poor performance on individual images.

The autoencoder has a limited training budget and may reconstruct some defects accurately. The patch detector uses a small memory bank and simplified feature extraction. These choices keep the experiment manageable, but limit what can be concluded about either method in general.

The next extension is the training-fraction study. After that, an independent dataset would be useful for checking whether the findings transfer. If I change the method after inspecting test failures, I need fresh evaluation data before claiming an improvement in generalisation.

## Saved outputs

The final cell creates `anomaly_results.zip`, containing metric tables, predictions, training history, split records, figures and checkpoints. The executed notebook is downloaded separately from Kaggle. I retain the run settings with the results so the comparison can be repeated.

## Data and references

MVTec AD is released under CC BY-NC-SA 4.0. This is a noncommercial educational experiment. The dataset and its authors should be credited in any accompanying report; the raw images are not included in the project repository.

- Bergmann et al. (2019), *MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection*. [Dataset and licence](https://www.mvtec.com/research-teaching/datasets/mvtec-ad).
- Roth et al. (2022), *Towards Total Recall in Industrial Anomaly Detection*. [PatchCore paper](https://openaccess.thecvf.com/content/CVPR2022/html/Roth_Towards_Total_Recall_in_Industrial_Anomaly_Detection_CVPR_2022_paper.html).
- [Torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/anomaly_results", "zip", root_dir=OUT)
print("Results and checkpoints:", archive)
print("Remember to download the executed .ipynb separately.")
